In [ ]:
# Step 3: Data Cleaning and Preprocessing

## Objectives

# 1. Load the three approved datasets.
# 2. Standardize column names.
# 3. Remove unnecessary whitespace from text values.
# 4. Identify empty and missing values.
# 5. Check duplicate records.
# 6. Preserve customer identifiers.
# 7. Save cleaned copies in the processed folder.
# 8. Generate a data-cleaning report.

# Important:
# - Original datasets must remain unchanged.
# - Banking and telecom datasets must remain separate.
# - No records will be removed without reviewing the audit results.

In [13]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [14]:
# Paste the path you copied from VS Code here
DATA_RAW = Path(
    r"D:\Dynamic_AI_Chatbot_Project\Raw_datasets"
)

# Project folder is the parent of Raw_datasets
PROJECT_ROOT = DATA_RAW.parent

# Output folders
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

REPORTS = PROJECT_ROOT / "outputs" / "evaluation_reports"

# Create folders if they do not exist
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_ROOT)
print("Dataset folder:", DATA_RAW)
print("Processed folder:", DATA_PROCESSED)
print("Reports folder:", REPORTS)

Project folder: D:\Dynamic_AI_Chatbot_Project
Dataset folder: D:\Dynamic_AI_Chatbot_Project\Raw_datasets
Processed folder: D:\Dynamic_AI_Chatbot_Project\data\processed
Reports folder: D:\Dynamic_AI_Chatbot_Project\outputs\evaluation_reports


In [19]:
print("FILES INSIDE Raw_datasets")
print("-" * 50)

for file in DATA_RAW.iterdir():
    print(repr(file.name))

FILES INSIDE Raw_datasets
--------------------------------------------------
'Customer%20Churn.csv'
'Customer-Churn-Records(1).csv'
'Telco_customer_churn(1).xlsx'


In [20]:
# Main telecom Excel dataset
telecom_excel_path = (
    DATA_RAW / "Telco_customer_churn(1).xlsx"
)

# Banking dataset
banking_csv_path = (
    DATA_RAW / "Customer-Churn-Records(1).csv"
)

# Second telecom dataset
telecom_csv_path = (
    DATA_RAW / "Customer%20Churn.csv"
)

# Verify all three paths
dataset_paths = {
    "Main Telecom Excel": telecom_excel_path,
    "Banking CSV": banking_csv_path,
    "Second Telecom CSV": telecom_csv_path
}

print("DATASET PATH VERIFICATION")
print("-" * 50)

for name, path in dataset_paths.items():

    print("\n", name)
    print("File:", path.name)
    print("Exists:", path.exists())

DATASET PATH VERIFICATION
--------------------------------------------------

 Main Telecom Excel
File: Telco_customer_churn(1).xlsx
Exists: True

 Banking CSV
File: Customer-Churn-Records(1).csv
Exists: True

 Second Telecom CSV
File: Customer%20Churn.csv
Exists: True


In [21]:
# Function to read CSV files safely
def read_csv_safely(file_path):

    encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]

    for encoding in encodings:

        try:
            return pd.read_csv(
                file_path,
                encoding=encoding
            )

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"Unable to read CSV file: {file_path}"
    )


# Load main telecom Excel
telecom_excel = pd.read_excel(
    telecom_excel_path,
    engine="openpyxl"
)

# Load banking CSV
banking_data = read_csv_safely(
    banking_csv_path
)

# Load second telecom CSV
telecom_csv = read_csv_safely(
    telecom_csv_path
)


print("ALL THREE DATASETS LOADED SUCCESSFULLY")

print("\nMain Telecom:", telecom_excel.shape)

print("Banking:", banking_data.shape)

print("Second Telecom:", telecom_csv.shape)

ALL THREE DATASETS LOADED SUCCESSFULLY

Main Telecom: (7043, 33)
Banking: (10000, 17)
Second Telecom: (7043, 21)


In [22]:
telecom_excel_clean = telecom_excel.copy(deep=True)

banking_data_clean = banking_data.copy(deep=True)

telecom_csv_clean = telecom_csv.copy(deep=True)

print("Working copies created successfully.")

Working copies created successfully.


In [23]:
# Standardize column names

def standardize_column_names(df):

    df = df.copy()

    cleaned_columns = []

    for column in df.columns:

        column = str(column).strip().lower()

        column = re.sub(
            r"[^a-z0-9]+",
            "_",
            column
        )

        column = column.strip("_")

        cleaned_columns.append(column)

    if len(cleaned_columns) != len(set(cleaned_columns)):
        raise ValueError(
            "Duplicate column names found after cleaning."
        )

    df.columns = cleaned_columns

    return df


telecom_excel_clean = standardize_column_names(
    telecom_excel_clean
)

banking_data_clean = standardize_column_names(
    banking_data_clean
)

telecom_csv_clean = standardize_column_names(
    telecom_csv_clean
)

print("Column names standardized successfully.")

Column names standardized successfully.


In [24]:
# Clean unnecessary spaces

def clean_text_values(df):

    df = df.copy()

    for column in df.select_dtypes(
        include=["object", "string"]
    ).columns:

        df[column] = df[column].map(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

        df[column] = df[column].replace(
            r"^\s*$",
            pd.NA,
            regex=True
        )

    return df


telecom_excel_clean = clean_text_values(
    telecom_excel_clean
)

banking_data_clean = clean_text_values(
    banking_data_clean
)

telecom_csv_clean = clean_text_values(
    telecom_csv_clean
)

print("Text cleaning completed.")


Text cleaning completed.


In [25]:
# Check missing values

datasets_clean = {
    "Main Telecom": telecom_excel_clean,
    "Banking": banking_data_clean,
    "Second Telecom": telecom_csv_clean
}

for name, df in datasets_clean.items():

    print("\n", "=" * 50)
    print("MISSING VALUES:", name)
    print("=" * 50)

    missing_report = pd.DataFrame({
        "Missing_Count": df.isnull().sum(),
        "Missing_Percentage": (
            df.isnull().mean() * 100
        ).round(2)
    })

    display(missing_report)


MISSING VALUES: Main Telecom


,Missing_Count,Missing_Percentage
customerid,0,0.00
count,0,0.00
country,0,0.00
state,0,0.00
city,0,0.00
zip_code,0,0.00
lat_long,0,0.00
latitude,0,0.00
longitude,0,0.00
gender,0,0.00



MISSING VALUES: Banking


,Missing_Count,Missing_Percentage
customerid,0,0.0
surname,0,0.0
creditscore,0,0.0
location,0,0.0
gender,0,0.0
age,0,0.0
tenure,0,0.0
account_balance,0,0.0
numofproducts,0,0.0
hascrcard,0,0.0



MISSING VALUES: Second Telecom


,Missing_Count,Missing_Percentage
customerid,0,0.00
gender,0,0.00
seniorcitizen,0,0.00
partner,0,0.00
dependents,0,0.00
tenure,0,0.00
phoneservice,0,0.00
multiplelines,0,0.00
internetservice,0,0.00
onlinesecurity,0,0.00


In [26]:
# Check duplicate records

for name, df in datasets_clean.items():

    print("\nDataset:", name)

    print("Total rows:", len(df))

    print(
        "Duplicate rows:",
        df.duplicated().sum()
    )

    print(
        "Unique rows:",
        len(df.drop_duplicates())
    )


Dataset: Main Telecom
Total rows: 7043
Duplicate rows: 0
Unique rows: 7043

Dataset: Banking
Total rows: 10000
Duplicate rows: 0
Unique rows: 10000

Dataset: Second Telecom
Total rows: 7043
Duplicate rows: 0
Unique rows: 7043


In [27]:
# Save the three cleaned datasets

# Save main telecom Excel
telecom_excel_clean.to_excel(
    DATA_PROCESSED / "telecom_churn_cleaned.xlsx",
    index=False
)

# Save banking CSV
banking_data_clean.to_csv(
    DATA_PROCESSED / "banking_churn_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save second telecom CSV
telecom_csv_clean.to_csv(
    DATA_PROCESSED / "telecom_churn_second_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

print("All three cleaned datasets saved successfully.")

All three cleaned datasets saved successfully.


In [28]:
print("PROCESSED DATASET VERIFICATION")
print("-" * 50)

for file in DATA_PROCESSED.iterdir():

    print(file.name, "| Exists:", file.exists())

PROCESSED DATASET VERIFICATION
--------------------------------------------------
banking_churn_cleaned.csv | Exists: True
telecom_churn_cleaned.xlsx | Exists: True
telecom_churn_second_cleaned.csv | Exists: True
